# 04. Pandas tour

Welcome to the fourth exercise of the course. The first three notebooks worked with pure Python: built in types, simple control flow, a small handful of standard library imports. This notebook brings in pandas, the workhorse library for tabular data in Python, and walks through the operations a working Python user reaches for almost every day when handling data: reading a file, inspecting it, slicing it, transforming it, grouping it, and dealing with missing values.

This exercise is denser than the previous three. Plan on closer to fifty minutes than thirty if pandas is new ground; readers who already use it daily will move through faster, but the bonus challenge at the end is worth lingering over.

Pandas is assumed to be installed (`pip install pandas`); no other installation is needed. The exercise does not connect to a TM1 server and does not use tm1py, but the data it works with is the kind a tm1py driven workflow produces all the time: monthly sales by region, with a few values missing here and there.

The task is a small tour of pandas, in seven steps:

1. **Read.** Load the CSV file `sales.csv` (sitting next to this notebook) into a DataFrame using `pd.read_csv`.
2. **Inspect.** Print the DataFrame's shape, columns, and index, then `head`, `tail`, `info`, and `describe` so the structure of the data is visible from inside the notebook.
3. **Extract.** Select a single column, a single row, a block of rows, a block of columns, and a combined row and column block. The two indexers `.iloc` (by integer position) and `.loc` (by label) both feature.
4. **Transform.** Add a `margin` column equal to `sales - cost`, using `.assign`.
5. **Handle missing values.** Count NaN per column, then produce two derived DataFrames: one with NaN rows dropped (`dropna`) and one with NaN values filled (`fillna`).
6. **Group.** Compute total sales per region with `groupby`. Then compute mean sales by region and year using a `groupby` with two keys.
7. **Extra challenge.** Build a pivot table with months as rows and regions as columns, showing total sales. The default row order will be alphabetic; reindex the rows to calendar order so the table reads naturally.

The notebook is arranged in four tiers, ordered from least to most support:

- **Pro.** The brief above is the whole instruction. A blank code cell follows the data, and the rest is yours.
- **Advanced.** A short description of the approach and the operations used, plus a code cell with a comment outline for each step.
- **Beginner.** Seven collapsible hints, then a code cell with the structure of the work in place. Slots marked `<...>` are pandas method names to fill in; the descriptive slot names suggest which method belongs in each spot.
- **Solution.** The full working code with intermediate prints at every step, plus an explanation of the choices it makes and an invitation to step through the result in a debugger.

Read the notebook from top to bottom. Start at whichever tier feels right today, drop down a tier if a higher one stalls, and treat the Solution as a reference rather than as a finish line. The Solution in this exercise is longer than in the earlier ones; reading it line by line is itself instructive.

Before retrying the exercise at a different tier in the same session, restart the Jupyter kernel and run the Data cell again. The Data cell re-imports pandas and re-binds `CSV_PATH`, and any DataFrame names defined at one tier would otherwise carry into the next.

Documentation pointers:

- pandas user guide: https://pandas.pydata.org/docs/user_guide/index.html
- `pd.read_csv`: https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
- Indexing and selecting data: https://pandas.pydata.org/docs/user_guide/indexing.html
- Group by user guide: https://pandas.pydata.org/docs/user_guide/groupby.html
- Working with missing data: https://pandas.pydata.org/docs/user_guide/missing_data.html
- Reshaping and pivot tables: https://pandas.pydata.org/docs/user_guide/reshaping.html

The exercise does not import numpy directly, but pandas brings it in as a dependency.

## Data

The data is a small Sales Plan dataset for 2025 and 2026: monthly sales and costs for four regions (`North`, `South`, `East`, `West`). The file `sales.csv` sits next to this notebook, in the same directory. The Pro, Advanced, Beginner, and Solution code cells read it back in with `pd.read_csv`.

The full dataset is 96 rows (two years times twelve months times four regions) with five columns: `year`, `month`, `region`, `sales`, and `cost`. Sales rise slightly from January to December and a small year on year increase separates 2025 from 2026; costs sit at 70 percent of sales. A few values are deliberately missing so the `dropna` and `fillna` steps in the brief have something to find:

- `sales` for March 2025 in the South region.
- `sales` for August 2026 in the East region.
- `cost` for July 2025 in the West region.
- `cost` for February 2026 in the North region.

The cell below imports pandas and binds `CSV_PATH` to the file's location. If the kernel was restarted between tiers, run it again so `pd` and `CSV_PATH` are in scope before any tier cell.

In [ ]:
import pandas as pd

CSV_PATH: str = "sales.csv"

## Pro

This tier is for readers who want to take the brief cold and write the program from a blank cell.

The CSV file `sales.csv` sits next to this notebook, and the Data cell just above has imported pandas and bound `CSV_PATH` to the file's location. The brief is at the top of the notebook. Read it, decide on the order of operations, and write the program in the cell below. There is no scaffolding here, and that is the point of the tier.

If the cell stops being productive to stare at, scroll on to Advanced or Beginner. The lower tiers do not spoil the work that has already gone into this one.

## Advanced

This tier is for readers who can see the rough shape of the solution and want a small amount of structure to write into.

The work proceeds in seven blocks, in the order the brief lays them out. Each block is short on its own; the size of the exercise comes from the variety, not the difficulty of any one step.

**Reading and inspection.** `pd.read_csv(path)` returns a DataFrame, given a CSV file. `df.shape` is the `(rows, columns)` tuple; `df.columns` and `df.index` describe the labels in each axis. `df.head()` and `df.tail()` show the first and last five rows by default, and accept an integer to change the count. `df.info()` prints the schema, column dtypes, and counts of values that are not NaN. `df.describe()` returns a numeric summary (count, mean, std, min, quartiles, max) for each numeric column.

**Extraction.** A single column comes from `df["sales"]` and is returned as a Series. Multiple columns come from `df[["year", "month"]]` (note the inner list of names) and are returned as a DataFrame. Rows are selected with `.iloc` for integer positions or `.loc` for labels. `df.iloc[0]` is a single row by position; `df.iloc[2:5]` is a slice of three rows; `.iloc` slicing follows ordinary Python slicing, so the right endpoint is excluded. `df.loc[0:4, ["year", "month"]]` selects rows 0 through 4 inclusive (`.loc` slicing is inclusive of the right endpoint, which is one of the easier traps in pandas) and the named columns.

**Transformation.** `df.assign(new_column=expression)` returns a new DataFrame with the new column added; the original `df` is unchanged. Most pandas methods follow this pattern of returning a new object rather than modifying in place.

**Missing values.** `df.isna()` returns a DataFrame of booleans of the same shape as `df` (True where a NaN is, False otherwise). Calling `.sum()` on that gives the NaN count per column. `df.dropna()` returns a DataFrame with rows containing any NaN removed. `df.fillna(0.0)` returns a DataFrame with NaN replaced by the given value. None of these modify `df` in place.

**Grouping.** `df.groupby(key)` groups rows by the value of one or more columns. The chained `["sales"].sum()` selects the sales column and aggregates within each group. Grouping by multiple keys (`df.groupby(["region", "year"])`) returns a result whose index has multiple levels (a `MultiIndex` in pandas terminology).

**Bonus.** `df.pivot_table(index=..., columns=..., values=..., aggfunc=...)` reshapes the data: the unique values in the `index` argument become rows, the unique values in the `columns` argument become columns, and the `aggfunc` aggregates the `values` column inside each cell. Pivot tables sort their indexes alphabetically by default; `pivot.reindex(some_order)` rearranges rows to follow a given order. Months that do not appear in the data become rows of NaN in the result.

The cell below has a comment outline for each block. Fill in each block in turn, and print the intermediate result at the end of every block so the data flow is visible while the code runs.

In [ ]:
import pandas as pd

CSV_PATH: str = "sales.csv"

# 1. Load the CSV file at CSV_PATH into a DataFrame.

# 2. Inspect.
# Print shape, columns, index. Then head, tail, info, describe.

# 3. Extract.
# A single column.
# A single row by integer position.
# A block of rows by integer position.
# A block of columns by name.
# A combined row and column block using .loc (note: .loc slicing is inclusive of the right endpoint).

# 4. Transform.
# Add a 'margin' column equal to sales minus cost.

# 5. Handle missing values.
# Count NaN per column. Drop rows with any NaN. Fill NaN with 0.0.

# 6. Group.
# Total sales by region.
# Mean sales by region and year.

# 7. Extra challenge.
# Build a pivot table with months as rows, regions as columns, sum of sales as values.
# Reindex the rows to follow calendar order.

CALENDAR_MONTHS: list[str] = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                              "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]


## Beginner

This tier is for readers who want guided support: hints to read one at a time, and a code skeleton with the structure of the work already in place. Slots marked `<...>` are pandas method names to fill in; the descriptive slot names suggest which method belongs in each spot.

Open the hints below in order, only as far as needed.

<details><summary>1. How is a CSV file loaded into a DataFrame?</summary>

`pd.read_csv("path/to/file.csv")` reads a CSV file from disk and returns a DataFrame. The path can be absolute or relative to the working directory. Default behaviour treats the first row as column headers, which matches the layout of `sales.csv`.

</details>

<details><summary>2. What do head, tail, info, and describe show?</summary>

`df.head()` returns the first five rows as a DataFrame; `df.head(20)` returns the first twenty. `df.tail()` does the same at the end. `df.info()` prints a schema summary: columns, dtypes, counts of values that are not NaN, and memory use; note that it prints rather than returns. `df.describe()` returns a DataFrame of summary statistics for the numeric columns: count, mean, std, min, the 25, 50, and 75 quartiles, and max.

</details>

<details><summary>3. How are rows and columns selected?</summary>

A single column: `df["sales"]` returns a Series. Multiple columns: `df[["year", "month"]]` returns a DataFrame; the inner list of names is essential. Rows by integer position: `df.iloc[0]` for one row, `df.iloc[2:5]` for a slice. Rows by label: `df.loc[0]` for one row, `df.loc[0:4]` for a slice. `df.loc[rows, columns]` combines both axes: `df.loc[0:4, ["year", "sales"]]`. There is one trap: `.iloc` slicing is exclusive of the right endpoint (Python convention), but `.loc` slicing is inclusive of it. So `df.iloc[0:4]` selects four rows and `df.loc[0:4]` selects five.

</details>

<details><summary>4. How is a derived column added?</summary>

`df.assign(margin=df["sales"] - df["cost"])` returns a new DataFrame with a `margin` column added; the original `df` is unchanged. The expression on the right hand side is computed once and applied across rows. The same pattern works for any new column: `df.assign(name=expression, another=other_expression)` adds two columns at once.

</details>

<details><summary>5. How are missing values counted and handled?</summary>

`df.isna()` returns a DataFrame of booleans of the same shape as `df` (True where a NaN is, False otherwise). Calling `.sum()` on that gives the NaN count per column. `df.dropna()` returns a new DataFrame with any row containing a NaN removed. `df.fillna(value)` returns a new DataFrame with NaN replaced by the given value. None of these modify `df` in place.

</details>

<details><summary>6. How are values aggregated by group?</summary>

`df.groupby("region")["sales"].sum()` groups the rows by `region`, takes the `sales` column within each group, and sums it. The result is a Series indexed by region. Grouping by multiple keys uses a list: `df.groupby(["region", "year"])["sales"].mean()`. The result has an index with two levels. Other aggregation methods (`.mean()`, `.median()`, `.count()`, `.max()`, and so on) work the same way.

</details>

<details><summary>7. How is a pivot table built and reordered?</summary>

`df.pivot_table(index="month", columns="region", values="sales", aggfunc="sum")` reshapes the data: the unique values in `index` become rows, the unique values in `columns` become columns, and `aggfunc` aggregates `values` inside each cell. The default row order is alphabetic. To force calendar order, call `.reindex(order)` on the result with a list of month names; pandas reorders the rows to match. Labels in `order` that do not appear in the table become rows of NaN.

</details>

In [ ]:
import pandas as pd

CSV_PATH: str = "sales.csv"

# 1. Load.
df = pd.<METHOD_FOR_CSV_READ>(CSV_PATH)

# 2. Inspect.
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Index:", df.index)

print("\nHead:")
print(df.head())

print("\nTail:")
print(df.tail())

print("\nInfo:")
df.info()

print("\nDescribe:")
print(df.describe())

# 3. Extract.
sales = df["sales"]
print("\nSingle column 'sales', first 5:")
print(sales.head())

first_row = df.iloc[0]
print("\nFirst row:")
print(first_row)

rows_2_to_4 = df.iloc[2:5]
print("\nRows at positions 2..4:")
print(rows_2_to_4)

descriptive = df[["year", "month", "region"]]
print("\nThree columns by name, first 5 rows:")
print(descriptive.head())

block = df.loc[0:4, ["year", "month", "sales"]]
print("\nRow and column block via .loc (inclusive of the right endpoint):")
print(block)

# 4. Transform: add a margin column.
df_with_margin = df.<METHOD_FOR_ADD_COLUMN>(margin=df["sales"] - df["cost"])
print("\nWith margin column, first 5 rows:")
print(df_with_margin.head())

# 5. Missing values.
print("\nNaN counts per column:")
print(df.isna().sum())

df_dropped = df.<METHOD_FOR_DROP_NAN>()
print(f"\ndropna(): {len(df)} rows -> {len(df_dropped)} rows.")

df_filled = df.<METHOD_FOR_FILL_NAN>(0.0)
print("\nfillna(0.0), first 5 rows:")
print(df_filled.head())

# 6. Group.
total_by_region = df.<METHOD_FOR_GROUPBY>("region")["sales"].sum()
print("\nTotal sales by region:")
print(total_by_region)

mean_by_region_year = df.groupby(["region", "year"])["sales"].mean()
print("\nMean sales by region and year:")
print(mean_by_region_year)

# 7. Bonus: pivot table, reindexed in calendar order.
CALENDAR_MONTHS: list[str] = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                              "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

pivot = df.<METHOD_FOR_PIVOT>(index="month", columns="region", values="sales", aggfunc="sum")
pivot_calendar = pivot.<METHOD_FOR_REINDEX>(CALENDAR_MONTHS)
print("\nPivot, calendar order:")
print(pivot_calendar)

## Solution

The cell below holds a complete, working version of the exercise. Each step prints its result so the effect of the operation is visible; reading the printed output line by line is the point. As before, this is offered as a reference rather than as the only correct form; pandas almost always offers more than one way to achieve a given result.

The work proceeds in seven blocks: load, inspect, extract, transform, handle missing values, group, and the bonus pivot. A few notes on the choices made:

- **`.iloc` and `.loc` are both shown.** The temptation to mix them up disappears once the difference between by integer position and by label is internalized, but the trap that catches everyone at first is the slicing convention: `.iloc[0:4]` selects four rows (Python convention, right endpoint excluded), while `.loc[0:4]` selects five rows (pandas convention, right endpoint included). The exercise places the two side by side so the contrast is on the page.
- **New DataFrames, not in place modifications.** `.assign`, `.dropna`, `.fillna`, `.pivot_table`, and `.reindex` all return new DataFrames rather than modifying the original. This is pandas' default and is worth absorbing as a habit; it is what keeps long chains of operations debuggable.
- **The shorthand groupby pattern.** `df.groupby("region")["sales"].sum()` is the canonical form: group by a key, select the column to aggregate, then call the aggregation method. The longer form `df.groupby("region").agg({"sales": "sum"})` does the same thing and is useful when more than one column is being aggregated; for a single column the shorthand reads better.
- **The pivot reindex.** Alphabetic order is rarely the right default for an axis that represents time. The explicit `pivot.reindex(CALENDAR_MONTHS)` makes the intent visible, and it survives changes in the data: if a month happens to be missing from the source table for some reason, reindex puts a row of NaN in its place rather than silently dropping it from the report.

A note on running this code: the cell is long, and each step prints something. Read each block of output carefully before moving on; that is how a feel for what each operation does is built. Better still, copy the code into a debugger (the VS Code Python debugger, or `pdb` from the terminal) and step through it one line at a time. Pandas operations transform DataFrames in ways that are not always obvious from the function names alone, and seeing the shape, dtypes, and contents of each intermediate value is what cements the mental model. Set a watch on `df`, `df_with_margin`, `df_filled`, `total_by_region`, and `pivot_calendar`; the same data, reshaped six different ways, becomes intuitive after a single thoughtful pass through the debugger.

In [ ]:
import pandas as pd

CSV_PATH: str = "sales.csv"

# 1. Load.
df = pd.read_csv(CSV_PATH)

# 2. Inspect.
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Index:", df.index)

print("\nHead:")
print(df.head())

print("\nTail:")
print(df.tail())

print("\nInfo:")
df.info()

print("\nDescribe:")
print(df.describe())

# 3. Extract.
sales = df["sales"]
print("\nSingle column 'sales', first 5 values:")
print(sales.head())

first_row = df.iloc[0]
print("\nFirst row by position (df.iloc[0]):")
print(first_row)

rows_2_to_4 = df.iloc[2:5]
print("\nRows at positions 2..4 (df.iloc[2:5], right endpoint excluded):")
print(rows_2_to_4)

descriptive = df[["year", "month", "region"]]
print("\nThree columns by name (year, month, region), first 5 rows:")
print(descriptive.head())

block = df.loc[0:4, ["year", "month", "sales"]]
print("\nCombined block via .loc (rows 0..4 inclusive, three named columns):")
print(block)

# 4. Transform: add a margin column.
df_with_margin = df.assign(margin=df["sales"] - df["cost"])
print("\nWith margin column, first 5 rows:")
print(df_with_margin.head())

# 5. Handle missing values.
print("\nNaN counts per column:")
print(df.isna().sum())

df_dropped = df.dropna()
print(f"\ndropna(): {len(df)} rows -> {len(df_dropped)} rows.")

df_filled = df.fillna(0.0)
print("\nfillna(0.0), first 5 rows:")
print(df_filled.head())

# 6. Group.
total_by_region = df.groupby("region")["sales"].sum()
print("\nTotal sales by region:")
print(total_by_region)

mean_by_region_year = df.groupby(["region", "year"])["sales"].mean()
print("\nMean sales by region and year:")
print(mean_by_region_year)

# 7. Extra challenge: pivot, then reindex in calendar order.
CALENDAR_MONTHS: list[str] = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                              "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

pivot = df.pivot_table(index="month", columns="region", values="sales", aggfunc="sum")
print("\nPivot before reindex (alphabetic month order):")
print(pivot)

pivot_calendar = pivot.reindex(CALENDAR_MONTHS)
print("\nPivot reindexed to calendar order:")
print(pivot_calendar)